# Word2Vec Implementation (Skip-gram)
### Compatible with PyTorch 2.2.2+cpu

In [ ]:
from functools import partial
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch import optim

import torchtext.datasets as datasets
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator

print(f"PyTorch version: {torch.__version__}")

In [ ]:
# Setup device
if torch.cuda.is_available():
    device = torch.device('cuda', 0)
else:
    device = torch.device('cpu')

print(f"Using device: {device}")

In [ ]:
# Load IMDB dataset
print("Loading IMDB dataset...")
train_iter = datasets.IMDB(split='train')
train_data = list(train_iter)
print(f"Train data loaded: {len(train_data)} samples")

In [ ]:
eval_iter = datasets.IMDB(split='test')
eval_data = list(eval_iter)
print(f"Test data loaded: {len(eval_data)} samples")

In [ ]:
# Process training data - extract reviews only
print("Processing training data...")
mapped_train_data = [review for label, review in train_data]

# Uncomment to use subset for faster testing
# mapped_train_data = mapped_train_data[0:2000]
print(f"Processed {len(mapped_train_data)} training reviews")

In [ ]:
# Process test data
print("Processing test data...")
mapped_eval_data = [review for label, review in eval_data]

# Uncomment to use subset for faster testing
# mapped_eval_data = mapped_eval_data[0:2000]
print(f"Processed {len(mapped_eval_data)} test reviews")

In [ ]:
# Check data
print("Sample review:")
print(mapped_train_data[0][:200], "...")
print(f"\nType: {type(mapped_train_data[0])}")

In [ ]:
# Initialize tokenizer
tokenizer = get_tokenizer("basic_english")

In [ ]:
# Build vocabulary
min_word_freq = 20

def build_vocab(mapped_train_data, tokenizer):        
    vocab = build_vocab_from_iterator(
        map(tokenizer, mapped_train_data),
        specials=["<unk>"],
        min_freq=min_word_freq
    )
    vocab.set_default_index(vocab["<unk>"])
    return vocab

print("Building vocabulary...")
vocab = build_vocab(mapped_train_data, tokenizer)
print("Vocabulary built successfully")

In [ ]:
vocab_size = len(vocab)
print(f"Vocabulary size: {vocab_size}")

In [ ]:
# Hyperparameters
window_size = 4  # leads to context window size of 9
max_seq_len = 256
max_norm = 1
embed_dim = 300
batch_size = 16

# Text processing pipeline
text_pipeline = lambda x: vocab(tokenizer(x))

In [ ]:
# Test pipeline
sample = text_pipeline("Hello World")
print(f"Sample tokens: {sample}")
print(f"Type: {type(sample)}")

In [ ]:
def collate_skipgram(batch, text_pipeline):
    batch_input_word, batch_target_words = [], []
    
    for review in batch:
        review_tokens_ids = text_pipeline(review)

        if len(review_tokens_ids) < window_size * 2 + 1:
            continue

        if max_seq_len:
            review_tokens_ids = review_tokens_ids[:max_seq_len]

        for idx in range(len(review_tokens_ids) - window_size * 2):
            current_ids_sequence = review_tokens_ids[idx : (idx + window_size * 2 + 1)]
            input_word = current_ids_sequence.pop(window_size)
            target_words = current_ids_sequence

            for target_word in target_words:
                batch_input_word.append(input_word)
                batch_target_words.append(target_word)

    batch_input_word = torch.tensor(batch_input_word, dtype=torch.long)
    batch_target_words = torch.tensor(batch_target_words, dtype=torch.long)
    return batch_input_word, batch_target_words

In [ ]:
# Create DataLoaders
traindl_skipgram = DataLoader(
    mapped_train_data,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=partial(collate_skipgram, text_pipeline=text_pipeline)
)

evaldl_skipgram = DataLoader(
    mapped_eval_data,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=partial(collate_skipgram, text_pipeline=text_pipeline)
)

In [ ]:
class SkipGram(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embeddings = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            max_norm=max_norm
        )
        self.linear = nn.Linear(
            in_features=embed_dim,
            out_features=vocab_size,
        )

    def forward(self, x):
        x = self.embeddings(x)
        x = self.linear(x)
        return x

In [ ]:
def train_one_epoch(model, dataloader, opt, loss_fn):
    model.train()
    running_loss = []

    for i, batch_data in enumerate(dataloader):
        inputs = batch_data[0].to(device)
        targets = batch_data[1].to(device)
        
        opt.zero_grad()
        outputs = model(inputs)
        loss = loss_fn(outputs, targets)
        loss.backward()
        opt.step()

        running_loss.append(loss.item())

    epoch_loss = np.mean(running_loss)
    return epoch_loss

In [ ]:
def validate_one_epoch(model, dataloader, loss_fn):
    model.eval()
    running_loss = []

    with torch.no_grad():
        for i, batch_data in enumerate(dataloader):
            inputs = batch_data[0].to(device)
            targets = batch_data[1].to(device)

            outputs = model(inputs)
            loss = loss_fn(outputs, targets)

            running_loss.append(loss.item())

    epoch_loss = np.mean(running_loss)
    return epoch_loss

In [ ]:
# Initialize model and training
loss_fn = nn.CrossEntropyLoss()
n_epochs = 5

model = SkipGram(vocab_size).to(device)
opt = optim.Adam(params=model.parameters(), lr=0.001)

print(f"Training Skip-gram model on {device}...")
print(f"Vocabulary size: {vocab_size}")
print(f"Embedding dimension: {embed_dim}")

In [ ]:
# Training loop
for e in range(n_epochs):
    print(f"Epoch {e+1}/{n_epochs}", end=" ")
    train_loss = train_one_epoch(model, traindl_skipgram, opt, loss_fn)
    val_loss = validate_one_epoch(model, evaldl_skipgram, loss_fn)
    print(f"Train Loss: {train_loss:.3f}, Val Loss: {val_loss:.3f}")

In [ ]:
# Extract embeddings
trimmed_model = model.embeddings
print(trimmed_model)

In [ ]:
# Test similarity
print("Testing word similarities...")
print(f"Sample vocabulary: {vocab.get_itos()[0:100]}")
print(f"Indices for 'film' and 'movie': {vocab.lookup_indices(['film', 'movie'])}")

In [ ]:
# Calculate cosine similarity
emb1 = trimmed_model(torch.tensor(vocab.lookup_indices(["film"])).to(device))
emb2 = trimmed_model(torch.tensor(vocab.lookup_indices(["movie"])).to(device))
print(f"Embedding shapes: {emb1.shape}, {emb2.shape}")

cos = torch.nn.CosineSimilarity(dim=1)
print(f"Similarity between 'film' and 'movie': {cos(emb1, emb2).item():.4f}")

emb1 = trimmed_model(torch.tensor(vocab.lookup_indices(["his"])).to(device))
emb2 = trimmed_model(torch.tensor(vocab.lookup_indices(["from"])).to(device))
print(f"Similarity between 'his' and 'from': {cos(emb1, emb2).item():.4f}")

emb1 = trimmed_model(torch.tensor(vocab.lookup_indices(["he"])).to(device))
emb2 = trimmed_model(torch.tensor(vocab.lookup_indices(["were"])).to(device))
print(f"Similarity between 'he' and 'were': {cos(emb1, emb2).item():.4f}")